# Power-BI-chatbot — Google Colab launcher

Repository: https://github.com/vinay-kumar0901/Power-BI-chatbot

Run every cell from top to bottom in a fresh Google Colab runtime. When prompted, upload `Power-BI-chatbot.zip` (or the ZIP downloaded from the GitHub repository). The notebook installs dependencies and Ollama, downloads the model, starts the app, checks its health, and creates a temporary Cloudflare Quick Tunnel URL.


In [ ]:
from google.colab import files
uploaded = files.upload()
zip_names = [n for n in uploaded if n.lower().endswith('.zip')]
if not zip_names:
    raise ValueError('Upload Power-BI-chatbot.zip or a ZIP downloaded from the Power-BI-chatbot GitHub repository.')
# Prefer the explicitly named project archive when multiple ZIPs were uploaded.
selected = next((n for n in zip_names if n.lower() == 'power-bi-chatbot.zip'), zip_names[0])
ZIP_PATH = '/content/' + selected
print('Using project archive:', ZIP_PATH)


In [ ]:
import os, zipfile, shutil, glob, subprocess, sys
# Bound numerical-library threads to avoid CPU oversubscription.
os.environ['BI_GPT_DATA_WORKERS'] = str(min(os.cpu_count() or 2, 4))
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['NUMEXPR_NUM_THREADS'] = '1'

PROJECT_ROOT='/content/power_bi_chatbot'
shutil.rmtree(PROJECT_ROOT, ignore_errors=True)
os.makedirs(PROJECT_ROOT, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH) as z:
    base=os.path.realpath(PROJECT_ROOT)
    for item in z.infolist():
        dest=os.path.realpath(os.path.join(PROJECT_ROOT,item.filename))
        if dest != base and not dest.startswith(base+os.sep):
            raise ValueError('Unsafe ZIP path')
    z.extractall(PROJECT_ROOT)

req=glob.glob(PROJECT_ROOT+'/**/requirements.txt',recursive=True)
if len(req)!=1:
    raise RuntimeError(f'Expected one requirements.txt file, found: {req}')
PROJECT_DIR=os.path.dirname(req[0])
os.chdir(PROJECT_DIR)
subprocess.check_call([sys.executable,'-m','pip','install','-q','-r','requirements.txt'])
print('Project folder:', PROJECT_DIR)
print('Dependencies installed.')


In [ ]:
import subprocess, time, requests, os, shutil
# Install zstd only when the current runtime does not already have it.
missing = [pkg for pkg, cmd in [('zstd','zstd'), ('curl','curl')] if shutil.which(cmd) is None]
if missing:
    subprocess.run(['apt-get','install','-y','-qq',*missing], check=True)
if shutil.which('ollama') is None:
    inst=subprocess.run('curl -fsSL https://ollama.com/install.sh | sh',shell=True,capture_output=True,text=True)
    print(inst.stdout[-1200:]); print(inst.stderr[-2000:])
    if inst.returncode: raise RuntimeError('Ollama installation failed')
else:
    print('Ollama is already installed; skipping reinstall.')
# Reuse a live Ollama service when possible.
try:
    requests.get('http://127.0.0.1:11434/api/tags',timeout=3).raise_for_status()
    ollama_proc = None
    print('Ollama server already ready.')
except requests.RequestException:
    OLLAMA_LOG=open('/content/ollama.log','w',buffering=1)
    ollama_proc=subprocess.Popen(['ollama','serve'],stdout=OLLAMA_LOG,stderr=subprocess.STDOUT)
    for i in range(45):
        try:
            if requests.get('http://127.0.0.1:11434/api/tags',timeout=3).ok: break
        except requests.RequestException: pass
        time.sleep(2)
    else: raise RuntimeError(open('/content/ollama.log').read()[-5000:])
    print('Ollama server started and ready.')


In [ ]:
import subprocess, os, json
CHAT_MODEL='llama3.2:1b'
models=subprocess.run(['ollama','list'],capture_output=True,text=True,check=True).stdout
if CHAT_MODEL not in models:
    print('Downloading model once for this runtime:', CHAT_MODEL)
    p=subprocess.run(['ollama','pull',CHAT_MODEL],text=True)
    if p.returncode: raise RuntimeError('Model download failed')
else:
    print('Model already downloaded; skipping pull:', CHAT_MODEL)
with open(os.path.join(PROJECT_DIR,'.env'),'w') as f:
    f.write('OLLAMA_BASE_URL=http://127.0.0.1:11434\nOLLAMA_MODEL='+CHAT_MODEL+'\nRAG_MODE=tfidf\nOLLAMA_REQUEST_TIMEOUT=300\nOLLAMA_CONNECT_TIMEOUT=10\nOLLAMA_NUM_THREADS='+str(min(os.cpu_count() or 2,8))+'\nBI_GPT_DATA_WORKERS='+str(min(os.cpu_count() or 2,4))+'\n')
print('Fast model configured; CPU threads:', min(os.cpu_count() or 2,8))


In [ ]:
import subprocess, sys, os
os.chdir(PROJECT_DIR)
p=subprocess.run([sys.executable,'run_setup.py'],text=True,capture_output=True)
print(p.stdout[-6000:]); print(p.stderr[-3000:])
if p.returncode: raise RuntimeError('Project setup failed')

In [ ]:
import subprocess, sys, time, requests, os
APP_LOG=open('/content/bi_gpt_app.log','w',buffering=1)
app_proc=subprocess.Popen([sys.executable,'-u','-m','uvicorn','app.main:app','--host','0.0.0.0','--port','8000'],cwd=PROJECT_DIR,stdout=APP_LOG,stderr=subprocess.STDOUT)
for i in range(60):
    if app_proc.poll() is not None: raise RuntimeError(open('/content/bi_gpt_app.log').read()[-8000:])
    try:
        h=requests.get('http://127.0.0.1:8000/api/health',timeout=3)
        if h.ok: print('BI-GPT ready:',h.json()); break
    except requests.RequestException: pass
    time.sleep(2)
else: raise RuntimeError(open('/content/bi_gpt_app.log').read()[-8000:])

In [ ]:
import requests, os, subprocess
url='https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'
r=requests.get(url,timeout=120); r.raise_for_status()
CLOUDFLARED='/content/cloudflared'; open(CLOUDFLARED,'wb').write(r.content); os.chmod(CLOUDFLARED,0o755)
subprocess.run([CLOUDFLARED,'--version'],check=True)
print('cloudflared ready')

In [ ]:
import subprocess, time, re, requests, socket
logh=open('/content/cloudflared.log','w',buffering=1)
tunnel_proc=subprocess.Popen([CLOUDFLARED,'tunnel','--no-autoupdate','--protocol','http2','--url','http://127.0.0.1:8000'],stdout=logh,stderr=subprocess.STDOUT)
PUBLIC_URL=None
for i in range(90):
    if tunnel_proc.poll() is not None: break
    logh.flush()
    log=open('/content/cloudflared.log',errors='replace').read()
    m=re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com',log)
    if m:
        candidate=m.group(0)
        try:
            if requests.get(candidate+'/api/health',timeout=8).ok: PUBLIC_URL=candidate; break
        except requests.RequestException: pass
    time.sleep(2)
if not PUBLIC_URL:
    print(open('/content/cloudflared.log').read()[-8000:]); raise RuntimeError('Tunnel not verified; do not use an old URL')
print('BI-GPT public URL:',PUBLIC_URL)
print('Health:',requests.get(PUBLIC_URL+'/api/health',timeout=15).text)
print('Keep this runtime active while using the app.')

## Speed notes
- Smaller 1B model reduces model download and inference time; answers may be less accurate than 3B.
- TF-IDF retrieval avoids embedding model calls.
- SQL proposal generation is skipped for non-data questions.
- Choose a GPU runtime where available. Do not run repeated setup/model download cells if the current runtime is still active.
- A Quick Tunnel URL expires when its process/runtime stops. Always use the newly printed URL.


In [ ]:
# BI-GPT single-question evaluation (end-to-end)
# Change TEST_QUESTION to evaluate any ONE question. Default tests the Net Sales by category calculation.
import requests, time, json, re

TEST_QUESTION = "What is the Net Sales by categories?"
BASE_URL = PUBLIC_URL.rstrip('/')

# Select the first available dashboard and create a clean test conversation.
dashboards = requests.get(BASE_URL + '/api/dashboards', timeout=20).json()
if not dashboards:
    raise RuntimeError('No dashboard is registered. Upload a PBIP dashboard ZIP first.')
dash_name = next((d['name'] for d in dashboards if d['name'].lower() == 'bi project'), dashboards[0]['name'])
chat = requests.post(BASE_URL + '/api/chats', json={'dashboard': dash_name}, timeout=20)
chat.raise_for_status()
chat_id = chat.json()['id']

started = time.monotonic()
job = requests.post(BASE_URL + '/api/chat/start', json={
    'chat_id': chat_id, 'question': TEST_QUESTION, 'dashboard': dash_name
}, timeout=30)
job.raise_for_status()
job_id = job.json()['job_id']
print('Evaluating one question:', TEST_QUESTION)
print('Dashboard:', dash_name)

result = None
for _ in range(240):  # up to 8 minutes; Ollama may need a cold-start warm-up
    time.sleep(2)
    status = requests.get(BASE_URL + '/api/chat/status/' + job_id, timeout=20)
    status.raise_for_status()
    payload = status.json()
    if payload.get('status') == 'completed':
        result = payload['result']
        break
    if payload.get('status') == 'failed':
        raise RuntimeError(payload.get('error', 'Question evaluation failed'))
if result is None:
    raise TimeoutError('Model did not finish within 8 minutes. Check /content/ollama.log and /content/bi_gpt_app.log.')

elapsed = time.monotonic() - started
answer_text = result.get('answer', '')
print('\n' + '='*70)
print('MODEL ANSWER\n')
print(answer_text)
print('\n' + '='*70)
print(f'End-to-end response time: {elapsed:.1f} seconds')
print('SQL generated:', 'Yes' if result.get('sql') else 'No')
print('Sources retrieved:', len(result.get('sources', [])))
print('SQL reason:', result.get('sql_reason', ''))

# Lightweight evaluation checks; these inspect the returned answer, not a second LLM call.
checks = {
    'mentions Net Sales': bool(re.search(r'net\s*sales', answer_text, re.I)),
    'does not equate Net Sales with all operating expenses': not bool(re.search(r'net sales.{0,100}(all expenses|overheads)', answer_text, re.I|re.S)),
    'contains verified SQL result path': bool(result.get('sql')) and 'JOIN Products' in result.get('sql', ''),
    'returns category breakdown': bool(re.search(r'\|\s*Category\s*\|', answer_text, re.I)),
    'shows DAX discount calculation': 'Order Discount Rate' in answer_text and 'Line Discount' in answer_text,
}
print('\nEVALUATION')
for label, passed in checks.items():
    print(('PASS' if passed else 'CHECK') + ' - ' + label)
print('\nEdit TEST_QUESTION in this cell and rerun it to test a different single question.')


## Runtime and parallelism
- CSV/XLSX files are read concurrently with a bounded process pool; SQLite writes remain serialized for database safety.
- Chat retrieval, conversation-history loading, and schema loading run concurrently using threads.
- BLAS/OpenMP thread counts are capped to prevent CPU oversubscription.
- The final evaluation cell sends one test question through the live app.
- The included sample PBIP model defines its own Net Sales measure; verify numeric results against the Power BI model.
- Application version: 3.8.0.


## UI, security and public tunnel
- Assistant responses are rendered as safe Markdown in the browser: tables, headings, emphasis, lists, inline code and fenced SQL blocks.
- Model-generated HTML is escaped before formatting.
- The Cloudflare Quick Tunnel is public and temporary. Do not upload private, confidential or regulated data.
- Keep the Colab runtime active while using the app. If it stops, rerun the notebook and use the newly generated URL.
- GitHub repository: https://github.com/vinay-kumar0901/Power-BI-chatbot
